In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import sqlite3
BASE = '/content/drive/MyDrive/Akhila-portfolio/Project1-data/'
conn = sqlite3.connect(BASE + 'medicare.db')
claims = pd.read_sql_query('SELECT * FROM all_claims', conn)
bene = pd.read_sql_query('SELECT * FROM beneficiary', conn)
conn.close()
print(claims.shape, bene.shape)

(5598898, 8) (116352, 18)


In [3]:
claims.isna().sum()


,0
patient_id,0
claim_id,0
thru_dt,11321
paid_amt,0
diag1,5846
hcpcs1,264930
provider_id,9128
claim_type,0


In [4]:
print('duplicate claim rows within type:')
print(claims.duplicated(subset=['claim_type','claim_id']).sum())

duplicate claim rows within type:
11043


In [5]:
print('negative paid rows:', (claims['paid_amt'] < 0).sum())
print('zero paid rows:', (claims['paid_amt'] == 0).sum())
claims['paid_amt'].describe()

negative paid rows: 2621
zero paid rows: 511831


,paid_amt
count,5.598898e+06
mean,2.283286e+02
std,1.467909e+03
min,-8.000000e+03
25%,2.000000e+01
50%,5.000000e+01
75%,1.000000e+02
max,5.700000e+04


In [6]:
claims = claims.dropna(subset=['patient_id'])
claims['thru_date'] = pd.to_datetime(claims['thru_dt'].astype(str), format='%Y%m%d', errors='coerce')
claims['claim_year'] = claims['thru_date'].dt.year
claims['is_unpaid'] = (claims['paid_amt'] == 0).astype(int)
print(claims['claim_year'].value_counts().sort_index())

Series([], Name: count, dtype: int64)


In [7]:
print(claims['thru_dt'].dtype)
print(claims['thru_dt'].head(10).tolist())

float64
[20100313.0, 20090418.0, 20090902.0, 20090920.0, 20100701.0, 20080912.0, 20080922.0, 20100606.0, 20100612.0, 20100619.0]


In [8]:
claims = claims.dropna(subset=['patient_id'])
claims['thru_date'] = pd.to_datetime(claims['thru_dt'].astype(str).str.replace('.0', '', regex=False), format='%Y%m%d', errors='coerce')
claims['claim_year'] = claims['thru_date'].dt.year
claims['is_unpaid'] = (claims['paid_amt'] == 0).astype(int)
print(claims['claim_year'].value_counts().sort_index())

claim_year
2008.0    2023711
2009.0    2210988
2010.0    1352878
Name: count, dtype: int64


In [9]:
print('rows after cleaning:', len(claims))
print('missing thru dates:', claims['thru_dt'].isna().sum())

rows after cleaning: 5598898
missing thru dates: 11321


In [10]:
ip_only = claims[claims['claim_type'] == 'Inpatient']
ip_counts = ip_only.groupby('patient_id').size().rename('ip_stays')
patient = claims.groupby('patient_id').agg(
    total_paid=('paid_amt','sum'), claim_count=('claim_id','count')).reset_index()
patient = patient.merge(ip_counts, on='patient_id', how='left')
patient['ip_stays'] = patient['ip_stays'].fillna(0).astype(int)
patient['repeat_flag'] = (patient['ip_stays'] >= 2).astype(int)
patient.to_csv(BASE + 'patient_summary.csv', index=False)
claims.to_csv(BASE + 'cleaned_all_claims.csv', index=False)
print(patient['repeat_flag'].mean().round(4))
patient.head()

0.1529


,patient_id,total_paid,claim_count,ip_stays,repeat_flag
0,00013D2EFD8E45D1,4290.0,7,1,0
1,00016F745862898F,55280.0,44,4,1
2,0001FDD721E223DC,50.0,2,0,0
3,00021CA6FF03E670,90.0,3,0,0
4,00024B3D2352D2D0,1720.0,30,0,0
